In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn import tree 
from sklearn import ensemble 
from sklearn import metrics 
import optuna

In [ ]:
#Прочитаем обработанные данные
X_train = pd.read_csv('../data/X_train_scaled.csv')
X_test = pd.read_csv('../data/X_test_scaled.csv')

y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

In [ ]:
#Попробуем построить рандомный лес
#Инициализируем лес
rf = ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    min_samples_leaf=5,
    max_depth=10,
    random_state=42
)
#Обучаем лес
rf.fit(X_train, y_train)
#Делаем предсказание
y_pred = rf.predict(X_test)
#Смотрим метрики
print("accuracy на тестовом наборе: {:.2f}".format(rf.score(X_test, y_test)))
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.recall_score(y_test, y_pred)))

accuracy на тестовом наборе: 0.83
f1_score на тестовом наборе: 0.83


Случайный лес справляется с задачей лучше, чем модели в бейзлайне, попробуем градиентный бустинг

In [ ]:
#Инициализируем градиентный бустинг
gb = ensemble.GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=300,
    min_samples_leaf=5,
    max_depth=5,
    random_state=42
)
#Обучаем бустинг
gb.fit(X_train, y_train)
#Делаем предсказание
y_pred = gb.predict(X_test)
#Смотрим метрику
round(metrics.f1_score(y_test, y_pred), 2)

0.82

Совсем немного, но хуже, попробуем стекинг с логрегрессией, деревом и бустингом

In [ ]:
#Инициализируем регрессию
lr = LogisticRegression(
    solver='sag',
    random_state=42,
    max_iter=1000
)
#Инициализируем дерево
tree_model = tree.DecisionTreeClassifier(
    random_state=42,
    criterion='entropy'
)

In [ ]:
#Делаем стэккинг с мета-моделью логистической регрессией
stack = ensemble.StackingClassifier(
    estimators=[
        ('lr', lr),
        ('tree', tree_model),
        ('gb', gb)
    ],
    final_estimator=LogisticRegression()
)
#Обучаем стеккинг
stack.fit(X_train, y_train)
#делаем предсказание
y_pred = stack.predict(X_test)
#Смотрим метрику
round(metrics.precision_score(y_test, y_pred), 2)

0.82

Без изменений

In [ ]:
importances = gb.feature_importances_
feature_importance = pd.Series(
    gb.feature_importances_,
    index=X_train.columns
)

feature_importance.sort_values(ascending=False).head(3)

1     0.516837
13    0.121365
8     0.078709
dtype: float64

Попробуем найти гиперпараметры с помощью ***optuna***

In [ ]:
def optuna_rf(trial):
    #задаем пространства поиска гиперпараметров
    n_estimators = trial.suggest_int('n_estimators', 100, 200, 1)
    max_depth = trial.suggest_int('max_depth', 10, 30, 1)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 10, 1)

    #создаем модель
    model = ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    #обучаем модель
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = metrics.f1_score(y_test, y_pred)

    return f1

In [20]:
study = optuna.create_study(direction='maximize')
study.optimize(optuna_rf, n_trials=50)

[I 2026-03-07 13:02:37,750] A new study created in memory with name: no-name-77e25b79-ce16-4838-a6ff-0cd3850f4bee
[I 2026-03-07 13:02:38,187] Trial 0 finished with value: 0.8147912017851451 and parameters: {'n_estimators': 190, 'max_depth': 12, 'min_samples_leaf': 6}. Best is trial 0 with value: 0.8147912017851451.
[I 2026-03-07 13:02:38,512] Trial 1 finished with value: 0.807606973058637 and parameters: {'n_estimators': 125, 'max_depth': 18, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.8147912017851451.
[I 2026-03-07 13:02:38,864] Trial 2 finished with value: 0.8131168417701369 and parameters: {'n_estimators': 169, 'max_depth': 13, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.8147912017851451.
[I 2026-03-07 13:02:39,188] Trial 3 finished with value: 0.810553083280356 and parameters: {'n_estimators': 129, 'max_depth': 24, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.8147912017851451.
[I 2026-03-07 13:02:39,569] Trial 4 finished with value: 0.8106904231625836

In [21]:
print(study.best_params)

{'n_estimators': 144, 'max_depth': 13, 'min_samples_leaf': 5}


In [22]:
meta_model = ensemble.RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

meta_model.fit(X_train, y_train)
y_pred = meta_model.predict(X_test)

In [23]:
f1 = metrics.f1_score(y_test, y_pred)
accuracy = metrics.accuracy_score(y_test, y_pred)

print("F1:", round(f1, 2))
print("Accuracy:", round(accuracy, 2))

F1: 0.82
Accuracy: 0.83
